# Deepfake Detector v2 — DINOv2-ViT-B/14 (Local RTX 3050)
**Dataset:** `yangsangtai/tiny-genimage` via Kaggle API  
**Architecture:** DINOv2-ViT-B/14 → CLS token → LayerNorm → MLP head → binary logit  
**Training:** 3-phase progressive unfreezing · AdamW + CosineAnnealing · fp16 · grad accumulation  
**GPU:** RTX 3050 4 GB — batch 16 × 4 accum = effective batch 64  
**Output:** `deepfake_dinov2.pth` saved to `ml-service/models/`

## 1 · Verify GPU & Environment

In [6]:
import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'nvidia-smi not found')

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU      : {props.name}')
    print(f'VRAM     : {props.total_memory / 1e9:.1f} GB')
    torch.cuda.empty_cache()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing: {DEVICE}')
assert DEVICE == 'cuda', 'GPU not detected — check kernel is set to Python 3.11 (GPU)'

Sun May 17 02:34:16 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 546.30                 Driver Version: 546.30       CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  | 00000000:01:00.0  On |                  N/A |
| N/A   41C    P8               3W /  60W |   1117MiB /  4096MiB |      4%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

## 2 · Install Dependencies

In [7]:
%pip install -q transformers accelerate grad-cam scikit-learn matplotlib seaborn tqdm Pillow kaggle

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3 · Configuration
Edit here before running anything else.

In [8]:
import os
from pathlib import Path

# ── Kaggle dataset ─────────────────────────────────────────────────────────────
KAGGLE_DATASET = 'yangyufeng14/genimage'

# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('D:/deepfake project')
DATA_ROOT    = PROJECT_ROOT / 'data'
OUTPUT_DIR   = PROJECT_ROOT / 'outputs'
MODEL_OUT    = (PROJECT_ROOT / 'Expo' / 'deepfake-detector'
                / 'ml-service' / 'models' / 'deepfake_dinov2.pth')

# ── Image settings ─────────────────────────────────────────────────────────────
IMG_SIZE = 224

# ── RTX 3050 (4 GB VRAM) ───────────────────────────────────────────────────────
BATCH_SIZE       = 16
GRAD_ACCUM_STEPS = 4    # effective batch = 64
NUM_WORKERS      = 0    # must be 0 on Windows Jupyter

# ── Epochs per phase ───────────────────────────────────────────────────────────
P1_EPOCHS = 8
P2_EPOCHS = 15
P3_EPOCHS = 10

# ── Dataset size cap ───────────────────────────────────────────────────────────
# This Kaggle version of GenImage has ~14k images per class total —
# the cap is not needed. Set a number (e.g. 5_000) only for quick smoke tests.
MAX_IMAGES_PER_CLASS = None

# ──────────────────────────────────────────────────────────────────────────────
for d in [DATA_ROOT, OUTPUT_DIR, MODEL_OUT.parent]:
    d.mkdir(parents=True, exist_ok=True)

print('Config OK')
print(f'  Dataset      : {KAGGLE_DATASET}')
print(f'  Model out    : {MODEL_OUT}')
print(f'  Device       : {DEVICE}')
print(f'  Eff. batch   : {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}')

Config OK
  Dataset      : yangyufeng14/genimage
  Model out    : D:\deepfake project\Expo\deepfake-detector\ml-service\models\deepfake_dinov2.pth
  Device       : cuda
  Eff. batch   : 16 x 4 = 64


## 4 · Kaggle Setup & Dataset Download

**Before running this cell:**
1. Go to kaggle.com → Account → API → **Create New API Token** (downloads `kaggle.json`)
2. Place it at `C:/Users/EKTA/.kaggle/kaggle.json`
3. The file must contain: `{"username":"...","key":"..."}`

**Size warning:** `yangyufeng14/genimage` is the full GenImage dataset (~200 GB compressed).  
Make sure you have enough disk space. The download will take a while on a typical home connection.  
To re-download, delete `D:/deepfake project/data/kaggle_dataset/.downloaded`

In [9]:
import kaggle

dataset_dir = DATA_ROOT / 'kaggle_dataset'
dataset_dir.mkdir(parents=True, exist_ok=True)

marker = dataset_dir / '.downloaded'
if marker.exists():
    print(f'Dataset already present at {dataset_dir}')
    print('Delete .downloaded marker to re-download.')
else:
    print(f'Downloading {KAGGLE_DATASET} ...')
    print('This may take several minutes.\n')
    kaggle.api.authenticate()
    kaggle.api.dataset_download_files(
        KAGGLE_DATASET,
        path=str(dataset_dir),
        unzip=True,
        quiet=False,
    )
    marker.touch()
    print('\nDownload complete.')

print('\nTop-level contents:')
for p in sorted(dataset_dir.iterdir()):
    if p.name == '.downloaded':
        continue
    if p.is_dir():
        count = sum(1 for _ in p.rglob('*') if _.is_file())
        print(f'  [dir]  {p.name}  ({count:,} files)')
    else:
        print(f'  [file] {p.name}')

Dataset already present at D:\deepfake project\data\kaggle_dataset
Delete .downloaded marker to re-download.

Top-level contents:
  [dir]  imagenet_ai_0419_biggan  (5,000 files)
  [dir]  imagenet_ai_0419_vqdm  (5,000 files)
  [dir]  imagenet_ai_0424_sdv5  (5,000 files)
  [dir]  imagenet_ai_0424_wukong  (5,000 files)
  [dir]  imagenet_ai_0508_adm  (5,000 files)
  [dir]  imagenet_glide  (5,000 files)
  [dir]  imagenet_midjourney  (5,000 files)


## 5 · Discover Dataset Paths

GenImage structure (8 generator folders, each with train/val splits):
```
kaggle_dataset/
├── imagenet_ai_0419_biggan/
│   ├── train/  ai/  nature/
│   └── val/    ai/  nature/
├── imagenet_sdv4/
│   ├── train/  ai/  nature/      <- same nature/ images repeated
│   └── val/    ai/  nature/
└── ... (6 more generators)
```
Real images (`nature/`) are **identical** across all generator folders — we deduplicate by filename.  
Fake images (`ai/`) are pooled from all 8 generators so the model sees every generator type.

In [10]:
import random
from pathlib import Path
from collections import defaultdict

random.seed(42)
dataset_dir   = DATA_ROOT / 'kaggle_dataset'
IMG_EXTS      = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
SPLIT_ALIASES = {'valid': 'val', 'validation': 'val', 'training': 'train', 'testing': 'test'}
REAL_KEYS     = {'real', 'nature', 'authentic'}


def collect_images(folder):
    return [str(p) for p in Path(folder).rglob('*')
            if p.is_file() and p.suffix.lower() in IMG_EXTS]


# ── Aggregate across ALL generator folders ─────────────────────────────────────
# GenImage: N generator subfolders each with train/val → ai/ + nature/.
# Pool ai/ from all generators. Deduplicate nature/ by filename (same images repeated).

raw = defaultdict(lambda: {'real': {}, 'fake': []})
generators_found = set()

for item in sorted(dataset_dir.rglob('*')):
    if not item.is_dir():
        continue
    key = SPLIT_ALIASES.get(item.name.lower(), item.name.lower())
    if key not in ('train', 'val', 'test'):
        continue

    subdirs  = {d.name.lower(): d for d in item.iterdir() if d.is_dir()}
    real_dir = next((subdirs[k] for k in REAL_KEYS if k in subdirs), None)
    ai_dirs  = [d for name, d in subdirs.items() if name not in REAL_KEYS]

    if not real_dir and not ai_dirs:
        continue

    generators_found.add(item.parent.name)

    if real_dir:
        for p in collect_images(real_dir):
            fname = Path(p).name
            if fname not in raw[key]['real']:
                raw[key]['real'][fname] = p

    for d in ai_dirs:
        raw[key]['fake'].extend(collect_images(d))

# ── Build splits ───────────────────────────────────────────────────────────────
print(f'Generators found: {len(generators_found)}')
for g in sorted(generators_found):
    print(f'  {g}')

splits = {}
for split_key, data in raw.items():
    rp = list(data['real'].values())
    fp = data['fake']
    if rp and fp:
        splits[split_key] = {'real': rp, 'fake': fp}

print(f'\nSplit summary (before capping):')
for split, data in sorted(splits.items()):
    print(f'  {split:6s}: {len(data["real"]):>7,} unique real  '
          f'{len(data["fake"]):>7,} fake')

USE_PRESPLIT = bool(splits)

if USE_PRESPLIT:
    def load_split(key):
        if key not in splits:
            return [], []
        rp = splits[key]['real'][:]
        fp = splits[key]['fake'][:]
        if MAX_IMAGES_PER_CLASS:
            rp = random.sample(rp, min(MAX_IMAGES_PER_CLASS, len(rp)))
            fp = random.sample(fp, min(MAX_IMAGES_PER_CLASS, len(fp)))
        n  = min(len(rp), len(fp))
        rp = random.sample(rp, n)
        fp = random.sample(fp, n)
        return rp + fp, [0] * n + [1] * n

    train_paths, train_labels = load_split('train')

    if 'test' in splits:
        # Dataset has a genuine test split — use it directly
        val_paths,  val_labels  = load_split('val')
        test_paths, test_labels = load_split('test')
    else:
        # No test split — split val 50/50 into val and test
        print('\nNo test split found — splitting val 50/50 into val + test.')
        all_val_paths, all_val_labels = load_split('val')
        mid = len(all_val_paths) // 2
        # Shuffle before splitting so val and test are both mixed
        combined = list(zip(all_val_paths, all_val_labels))
        random.shuffle(combined)
        all_val_paths, all_val_labels = zip(*combined)
        val_paths,  val_labels  = list(all_val_paths[:mid]),  list(all_val_labels[:mid])
        test_paths, test_labels = list(all_val_paths[mid:]),  list(all_val_labels[mid:])

else:
    from sklearn.model_selection import train_test_split
    print('No pre-split — flat fallback...')
    all_real, all_fake = {}, []
    for item in dataset_dir.rglob('*'):
        if not item.is_dir():
            continue
        if any(k in item.name.lower() for k in REAL_KEYS):
            for p in collect_images(item):
                all_real[Path(p).name] = p
        elif item.name.lower() in ('ai', 'fake', 'generated'):
            all_fake.extend(collect_images(item))
    rp, fp = list(all_real.values()), all_fake
    assert rp and fp, f'Could not find images in {dataset_dir}'
    if MAX_IMAGES_PER_CLASS:
        rp = random.sample(rp, min(MAX_IMAGES_PER_CLASS, len(rp)))
        fp = random.sample(fp, min(MAX_IMAGES_PER_CLASS, len(fp)))
    n = min(len(rp), len(fp))
    rp, fp = random.sample(rp, n), random.sample(fp, n)
    all_paths, all_labels = rp + fp, [0]*n + [1]*n
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        all_paths, all_labels, test_size=0.2, stratify=all_labels, random_state=42)
    X_val, X_te, y_val, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)
    train_paths, train_labels = X_tr, y_tr
    val_paths,   val_labels   = X_val, y_val
    test_paths,  test_labels  = X_te, y_te

print(f'\nFinal split:')
print(f'  Train : {len(train_paths):>7,}  ({sum(train_labels):,} fake)')
print(f'  Val   : {len(val_paths):>7,}  ({sum(val_labels):,} fake)  ← early stopping monitor')
print(f'  Test  : {len(test_paths):>7,}  ({sum(test_labels):,} fake)  ← final held-out evaluation')

Generators found: 7
  imagenet_ai_0419_biggan
  imagenet_ai_0419_vqdm
  imagenet_ai_0424_sdv5
  imagenet_ai_0424_wukong
  imagenet_ai_0508_adm
  imagenet_glide
  imagenet_midjourney

Split summary (before capping):
  train :  14,000 unique real   14,000 fake
  val   :   3,500 unique real    3,500 fake

No test split found — splitting val 50/50 into val + test.

Final split:
  Train :  28,000  (14,000 fake)
  Val   :   3,500  (1,748 fake)  ← early stopping monitor
  Test  :   3,500  (1,752 fake)  ← final held-out evaluation


## 6 · Dataset & DataLoaders

In [11]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class DeepfakeDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths     = paths
        self.labels    = torch.tensor(labels, dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        return self.transform(img), self.labels[idx]


def make_loader(paths, labels, tf, shuffle):
    ds = DeepfakeDataset(paths, labels, tf)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )


train_loader = make_loader(train_paths, train_labels, train_tf, shuffle=True)
val_loader   = make_loader(val_paths,   val_labels,   val_tf,   shuffle=False)
test_loader  = make_loader(test_paths,  test_labels,  val_tf,   shuffle=False)

print(f'Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}')
print(f'Effective batch size: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}')

Batches — train: 1750, val: 219, test: 219
Effective batch size: 16 x 4 = 64


## 7 · Model — DINOv2-ViT-B/14

In [12]:
import torch.nn as nn
from transformers import Dinov2Model


class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = Dinov2Model.from_pretrained('facebook/dinov2-base')
        self.head = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.3),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, pixel_values):
        out = self.backbone(pixel_values=pixel_values)
        cls = out.last_hidden_state[:, 0]   # CLS token -> (B, 768)
        return self.head(cls)               # (B, 1) logits


model = DeepfakeDetector().to(DEVICE)
# gradient checkpointing is managed per-phase inside set_phase()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters : {total / 1e6:.1f}M')
print(f'Trainable  : {trainable / 1e6:.1f}M')

c:\Users\EKTA\PYTHON\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\EKTA\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Parameters : 86.8M
Trainable  : 86.8M


## 8 · Training Utilities

In [13]:
import copy
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

LABEL_SMOOTHING = 0.1


def smooth_bce(logits, targets):
    smooth = targets * (1 - LABEL_SMOOTHING) + 0.5 * LABEL_SMOOTHING
    return F.binary_cross_entropy_with_logits(logits.squeeze(1), smooth)


def set_phase(model, phase):
    """Freeze/unfreeze layers and toggle gradient checkpointing per phase."""
    for p in model.backbone.parameters():
        p.requires_grad_(False)
    for p in model.head.parameters():
        p.requires_grad_(True)

    if phase == 2:
        for layer in model.backbone.encoder.layer[-4:]:
            for p in layer.parameters():
                p.requires_grad_(True)
        for p in model.backbone.layernorm.parameters():
            p.requires_grad_(True)
    elif phase == 3:
        for p in model.backbone.parameters():
            p.requires_grad_(True)

    # Phase 1 backbone is frozen — checkpointing would double-run the forward
    # pass (use_reentrant=False in PyTorch 2.x) for zero benefit, wasting VRAM.
    if phase == 1:
        model.backbone.gradient_checkpointing_disable()
        print('Gradient checkpointing: OFF (backbone frozen)')
    else:
        model.backbone.gradient_checkpointing_enable()
        print('Gradient checkpointing: ON')

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Phase {phase} — trainable: {trainable / 1e6:.1f}M / {total / 1e6:.1f}M')


def train_epoch(model, loader, optimizer, scaler, accum_steps):
    model.train()
    loss_sum, n_correct, n_total = 0.0, 0, 0
    all_probs, all_labels = [], []
    optimizer.zero_grad()

    for step, (imgs, labels) in enumerate(tqdm(loader, desc='  train', leave=False)):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        with torch.amp.autocast('cuda'):
            logits = model(imgs)
            loss   = smooth_bce(logits, labels) / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        with torch.no_grad():
            probs      = torch.sigmoid(logits.squeeze(1)).detach().cpu()
            preds      = (probs > 0.5).float()
            loss_sum  += loss.item() * accum_steps * len(labels)
            n_correct += (preds == labels.cpu()).sum().item()
            n_total   += len(labels)
            all_probs.extend(probs.numpy())
            all_labels.extend(labels.cpu().numpy())

    return loss_sum / n_total, n_correct / n_total, roc_auc_score(all_labels, all_probs)


@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    loss_sum, n_correct, n_total = 0.0, 0, 0
    all_probs, all_labels = [], []

    for imgs, labels in tqdm(loader, desc='  eval ', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
            loss   = F.binary_cross_entropy_with_logits(logits.squeeze(1), labels)
        probs      = torch.sigmoid(logits.squeeze(1)).cpu()
        preds      = (probs > 0.5).float()
        loss_sum  += loss.item() * len(labels)
        n_correct += (preds == labels.cpu()).sum().item()
        n_total   += len(labels)
        all_probs.extend(probs.numpy())
        all_labels.extend(labels.cpu().numpy())

    return loss_sum / n_total, n_correct / n_total, roc_auc_score(all_labels, all_probs)


def run_phase(model, phase, epochs, lr, train_loader, val_loader, history):
    set_phase(model, phase)
    params    = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.amp.GradScaler('cuda')

    best_auc         = 0.0
    best_state       = None
    patience_counter = 0
    PATIENCE         = 5

    for epoch in range(1, epochs + 1):
        print(f'Epoch {epoch}/{epochs}')
        tr_loss, tr_acc, tr_auc = train_epoch(
            model, train_loader, optimizer, scaler, GRAD_ACCUM_STEPS)
        va_loss, va_acc, va_auc = eval_epoch(model, val_loader)
        scheduler.step()

        print(f'  train  loss={tr_loss:.4f}  acc={tr_acc:.4f}  auc={tr_auc:.4f}')
        print(f'  val    loss={va_loss:.4f}  acc={va_acc:.4f}  auc={va_auc:.4f}')

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)
        history['train_auc'].append(tr_auc)
        history['val_auc'].append(va_auc)
        history['phase'].append(phase)

        if va_auc > best_auc:
            best_auc   = va_auc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print(f'  Best auc={best_auc:.4f}')
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'  Early stop (patience={PATIENCE})')
                break

    model.load_state_dict(best_state)
    print(f'\nPhase {phase} done. Best val AUC: {best_auc:.4f}\n')
    return best_auc


history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'train_auc':  [], 'val_auc':  [],
    'phase':      [],
}
print('Utilities ready.')

Utilities ready.


## 9 · Phase 1 — Head Only (lr = 1e-3)
Backbone fully frozen. Fast — only 0.2M params train.

In [ ]:
print('=' * 55)
print('PHASE 1 — head only  (backbone fully frozen)')
print('=' * 55)
run_phase(model, phase=1, epochs=P1_EPOCHS, lr=1e-3,
          train_loader=train_loader, val_loader=val_loader, history=history)

PHASE 1 — head only  (backbone fully frozen)
Gradient checkpointing: OFF (backbone frozen)
Phase 1 — trainable: 0.2M / 86.8M
Epoch 1/8


  train  loss=0.5225  acc=0.7693  auc=0.8550
  val    loss=0.4331  acc=0.8031  auc=0.8939
  Best auc=0.8939
Epoch 2/8


  train  loss=0.4710  acc=0.8201  auc=0.9008
  val    loss=0.3927  acc=0.8377  auc=0.9182
  Best auc=0.9182
Epoch 3/8


  train  loss=0.4387  acc=0.8465  auc=0.9245
  val    loss=0.3614  acc=0.8540  auc=0.9311
  Best auc=0.9311
Epoch 4/8


KeyboardInterrupt: 

## 10 · Phase 2 — Last 4 Blocks + Head (lr = 5e-5)

In [ ]:
print('=' * 55)
print('PHASE 2 — last 4 transformer blocks + head')
print('=' * 55)
run_phase(model, phase=2, epochs=P2_EPOCHS, lr=5e-5,
          train_loader=train_loader, val_loader=val_loader, history=history)

## 11 · Phase 3 — Full Fine-tune (lr = 1e-5)

In [ ]:
print('=' * 55)
print('PHASE 3 — all layers unfrozen')
print('=' * 55)
run_phase(model, phase=3, epochs=P3_EPOCHS, lr=1e-5,
          train_loader=train_loader, val_loader=val_loader, history=history)

## 12 · Training History

In [ ]:
import matplotlib.pyplot as plt

epochs_total  = list(range(1, len(history['train_loss']) + 1))
phase_changes = [
    i + 1 for i in range(len(history['phase']) - 1)
    if history['phase'][i] != history['phase'][i + 1]
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (tr_key, va_key), title in zip(
    axes,
    [('train_loss', 'val_loss'), ('train_acc', 'val_acc'), ('train_auc', 'val_auc')],
    ['Loss (BCE)', 'Accuracy', 'AUC'],
):
    ax.plot(epochs_total, history[tr_key], label='train')
    ax.plot(epochs_total, history[va_key], label='val')
    for pc in phase_changes:
        ax.axvline(pc, color='gray', linestyle='--', alpha=0.5, label='phase change')
    ax.set(title=title, xlabel='Epoch')
    ax.legend()

plt.tight_layout()
out_path = str(OUTPUT_DIR / 'training_history.png')
plt.savefig(out_path, dpi=150)
print(f'Saved: {out_path}')
plt.show()

## 13 · Evaluation on Test Set

In [ ]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score, roc_curve)
import seaborn as sns

print('Evaluating on held-out test set...')
te_loss, te_acc, te_auc = eval_epoch(model, test_loader)
print(f'Test  loss={te_loss:.4f}  acc={te_acc:.4f}  auc={te_auc:.4f}')

model.eval()
y_prob, y_true_list = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Predict'):
        imgs = imgs.to(DEVICE)
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
        y_prob.extend(torch.sigmoid(logits.squeeze(1)).cpu().numpy())
        y_true_list.extend(labels.numpy())

y_prob = np.array(y_prob)
y_true = np.array(y_true_list)
y_pred = (y_prob > 0.5).astype(int)

print('\n' + classification_report(y_true, y_pred, target_names=['Real', 'Fake']))
print(f'ROC-AUC : {roc_auc_score(y_true, y_prob):.4f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'], ax=axes[0])
axes[0].set(title='Confusion Matrix', xlabel='Predicted', ylabel='Actual')

fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[1].plot(fpr, tpr, lw=2, color='darkorange', label=f'AUC={te_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set(title='ROC Curve', xlabel='FPR', ylabel='TPR')
axes[1].legend()

axes[2].hist(y_prob[y_true == 0], bins=60, alpha=0.6, color='steelblue', label='Real')
axes[2].hist(y_prob[y_true == 1], bins=60, alpha=0.6, color='tomato',    label='Fake')
axes[2].axvline(0.5, color='black', linestyle='--')
axes[2].set(title='Score Distribution', xlabel='P(fake)', ylabel='Count')
axes[2].legend()

plt.tight_layout()
out_path = str(OUTPUT_DIR / 'evaluation.png')
plt.savefig(out_path, dpi=150)
print(f'Saved: {out_path}')
plt.show()

## 14 · GradCAM++ Visualisation

In [ ]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget


def reshape_transform_vit(tensor, height=16, width=16):
    result = tensor[:, 1:, :]                                               # drop CLS
    result = result.reshape(result.size(0), height, width, result.size(2))
    return result.permute(0, 3, 1, 2).contiguous()


target_layer = model.backbone.encoder.layer[-1].norm1

cam = GradCAMPlusPlus(
    model=model,
    target_layers=[target_layer],
    reshape_transform=reshape_transform_vit,
)

inv_norm = transforms.Normalize(
    mean=[-m / s for m, s in zip(MEAN, STD)],
    std=[1 / s for s in STD],
)

n_show  = 8
indices = random.sample(range(len(test_paths)), n_show)

fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 7))

for col, idx in enumerate(indices):
    img_t  = val_tf(Image.open(test_paths[idx]).convert('RGB'))
    label  = test_labels[idx]
    img_in = img_t.unsqueeze(0).to(DEVICE)

    grayscale = cam(input_tensor=img_in, targets=[BinaryClassifierOutputTarget(0)])
    grayscale = grayscale[0]

    img_show = inv_norm(img_t).permute(1, 2, 0).numpy().clip(0, 1)
    overlay  = show_cam_on_image(img_show, grayscale, use_rgb=True)

    prob  = float(torch.sigmoid(model(img_in)).squeeze())
    title = f"{'Fake' if label else 'Real'} | p={prob:.2f}"

    axes[0, col].imshow(img_show); axes[0, col].axis('off')
    axes[0, col].set_title(title, fontsize=8)
    axes[1, col].imshow(overlay);  axes[1, col].axis('off')
    axes[1, col].set_title('GradCAM++', fontsize=8)

plt.tight_layout()
out_path = str(OUTPUT_DIR / 'gradcam_samples.png')
plt.savefig(out_path, dpi=150)
print(f'Saved: {out_path}')
plt.show()

## 15 · Export Model
Saves `deepfake_dinov2.pth` directly into `ml-service/models/`.  
Run the evaluation cell first so `y_true` / `y_prob` are available.

In [ ]:
from sklearn.metrics import roc_auc_score

torch.save({
    'model_state_dict': model.state_dict(),
    'test_auc':         float(roc_auc_score(y_true, y_prob)),
    'img_size':         IMG_SIZE,
    'architecture':     'dinov2-base',
}, str(MODEL_OUT))

size_mb = MODEL_OUT.stat().st_size / 1e6
print(f'Model saved : {MODEL_OUT}  ({size_mb:.0f} MB)')

for fname in ('training_history.png', 'evaluation.png', 'gradcam_samples.png'):
    src = OUTPUT_DIR / fname
    if src.exists():
        print(f'Output      : {src}')

print('\nDone. Update main.py to load the new weights.')